# LangChain 기반 맞춤형 콘서트 예매 가이드

NOL Ticket 상품 URL과 사용자 질문을 입력받아 HTML과 상세 이미지 OCR 원문을 수집하고, LangChain RAG 파이프라인으로 질문에 필요한 예매 정보만 구조화하여 답한다.

## 1. 주제 선정 — 누구의 어떤 불편을 해결하는가

**대상 사용자:** 긴 콘서트 예매 공지에서 자신의 상황에 필요한 정보만 빠르게 확인하고 싶은 관람객

NOL Ticket 예매 공지는 선예매, 본인 확인, 티켓 수령, 가격, 입장 규칙 등의 정보가
HTML 본문과 상세 이미지에 흩어져 있다. 사용자는 긴 공지를 직접 읽으며 자신의 상황과
관련된 내용을 찾아 조합해야 한다.

이 도우미는 **NOL Ticket 상품 URL과 자연어 질문**을 받아,
HTML과 상세 이미지 OCR 원문 중 질문과 관련된 Context만 검색하고
사용자 상황에 맞는 예매 정보로 구조화한다.

### 왜 단순 검색이 아니라 LLM인가?

`가격`, `배송`처럼 한 단어로 찾을 수 있는 질문은 키워드 검색만으로도 처리할 수 있다.
하지만 다음처럼 여러 조건이 섞인 질문은 서로 다른 공지 영역을 함께 해석해야 한다.

> 팬클럽 선예매를 하려면 언제 인증하고 언제 예매해야 해?

이 질문에는 **팬클럽 인증 기간 + 인증 조건 + 선예매 일정**을 함께 찾아 연결해야 한다.
따라서 본 프로젝트에서는 Retriever가 관련 Context를 찾고,
LLM이 사용자의 자연어 의도와 여러 공지 조각을 종합해 답변하도록 구성했다.

> 실제 서비스에서는 URL과 질문을 사용자 입력으로 받는다.
> 이 `.ipynb`는 보고서 재현성을 위해 하나의 URL과 대표 질문을 코드에 고정하여 실행한다.

## 2. 문제 해결 흐름 — 입력에서 출력까지

`고정 URL/질문 → HTML·OCR 수집 → Document → Chunk → Embedding/Chroma → Query Understanding → Retrieval/Reranking → Prompt + LLM → Structured Output`

대표 입력 1개는 아래 셀을 따라 전체 흐름을 확인하고,
마지막에는 **같은 URL에서 질문만 바꾼 3개 입력**을 추가 실행하여
질문에 따라 검색되는 Context와 최종 답변이 어떻게 달라지는지 비교한다.


## 0. 패키지 설치

Colab 런타임마다 검증된 버전을 설치한다. 설치가 끝나면 `런타임 → 세션 다시 시작`을 한 번 실행한 뒤 1번 셀부터 계속한다.

In [16]:
%pip install -q \
    "numpy==2.2.6" \
    "paddlepaddle==3.2.0" \
    "paddleocr==3.7.0" \
    "pillow==11.3.0" \
    "requests==2.34.2" \
    "beautifulsoup4==4.15.0" \
    "brotli==1.2.0" \
    "python-dotenv" \
    "langchain==1.4.2" \
    "langchain-core==1.6.3" \
    "langchain-openai==1.6.2" \
    "openai==3.16.2" \
    "httpx2==2.13.0" \
    "langchain-chroma==1.1.0" \
    "langchain-text-splitters==1.1.2"

# 1. 환경변수와 Chat Model

현재 폴더의 `.env`에서 `OPENAI_API_KEY`를 읽는다. 예매 정보는 일관성과 근거성이 중요하므로 `temperature=0`으로 설정한다.

In [17]:
import os
from pathlib import Path
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv(override=True)
PADDLE_CACHE = Path('/content/paddlex_cache')
os.environ.setdefault('PADDLE_PDX_CACHE_HOME', str(PADDLE_CACHE))
if not os.getenv('OPENAI_API_KEY'):
    raise RuntimeError('.env 파일에 OPENAI_API_KEY를 설정해 주세요.')

model = init_chat_model(
    'gpt-4o-mini',
    model_provider='openai',
    temperature=0,
    timeout=30,
    max_tokens=700,
    max_retries=1,
)
print('Paddle 모델 캐시:', PADDLE_CACHE)
print('Chat Model 초기화 완료:', model.model_name)

Paddle 모델 캐시: /content/paddlex_cache
Chat Model 초기화 완료: gpt-4o-mini


## 1-1. 실행 환경 검증

설치 셀에서 지정한 버전이 실제 Colab 런타임에 적용됐는지 확인한다. 버전이 다르면 0번 셀 실행 후 세션을 다시 시작해야 한다.

In [18]:
import sys
from importlib.metadata import version

import numpy
import paddle
import paddleocr

expected_versions = {
    'numpy': '2.2.6',
    'paddle': '3.2.0',
    'paddleocr': '3.7.0',
    'brotli': '1.2.0',
    'openai': '3.16.2',
    'httpx2': '2.13.0',
}
actual_versions = {
    'python': sys.version.split()[0],
    'numpy': numpy.__version__,
    'paddle': paddle.__version__,
    'paddleocr': paddleocr.__version__,
    'brotli': version('brotli'),
    'openai': version('openai'),
    'httpx2': version('httpx2'),
}

print('실행 Python:', sys.executable)
print('환경 정보:', actual_versions)

for package, expected_version in expected_versions.items():
    actual_version = actual_versions[package]
    if actual_version != expected_version:
        raise RuntimeError(
            f'{package} 버전 불일치: '
            f'expected={expected_version}, actual={actual_version}. '
            '0번 셀 실행 후 세션을 다시 시작해 주세요.'
        )

paddle.utils.run_check()

실행 Python: /usr/bin/python3
환경 정보: {'python': '3.13.15', 'numpy': '2.2.6', 'paddle': '3.2.0', 'paddleocr': '3.7.0', 'brotli': '1.2.0', 'openai': '3.16.2', 'httpx2': '2.13.0'}
Running verify PaddlePaddle program ... 
PaddlePaddle works well on 1 CPU.
PaddlePaddle is installed successfully! Let's start deep learning with PaddlePaddle now.


/usr/local/lib/python3.13/dist-packages/paddle/pir/math_op_patch.py:219: UserWarning: Value do not have 'place' interface for pir graph mode, try not to use it. None will be returned.
  warnings.warn(


# 3. 보고서용 고정 입력과 URL 검증

실제 서비스에서는 사용자가 NOL Ticket 상품 URL과 질문을 입력한다. 이 노트북은 **같은 조건으로 결과를 재현하기 위한 보고서**이므로 `concert_id`와 대표 질문을 코드에 고정한다.

현재 MVP는 `https://nol.yanolja.com/ticket/products/{concert_id}` 형식만 지원한다.

예제 링크
- https://nol.yanolja.com/ticket/products/26013161
- https://nol.yanolja.com/ticket/products/26013132 : 이미지 0장
- https://nol.yanolja.com/ticket/products/26012624
- https://nol.yanolja.com/ticket/products/26012865


In [19]:
concert_id = '26013161'
if not concert_id.isdigit():
    raise ValueError('concert_id는 숫자로만 입력해 주세요.')

page_url = f'https://nol.yanolja.com/ticket/products/{concert_id}'
print('concert_id:', concert_id)
print('NOL Ticket 상품 URL:', page_url)

concert_id: 26013161
NOL Ticket 상품 URL: https://nol.yanolja.com/ticket/products/26013161


# 3. HTML Loader

`requests + BeautifulSoup`으로 상품 페이지를 읽는다. 공연마다 달라지는 장소·가격·팬클럽명으로 문장을 선별하지 않고 정제된 HTML 원문 전체를 보존한다. 상세 공지 이미지는 NOL Ticket이 사용하는 이미지 호스트와 경로를 기준으로 탐지한다.

In [20]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

HEADERS = {'User-Agent': 'Mozilla/5.0', 'Accept-Language': 'ko-KR,ko;q=0.9'}

response = requests.get(page_url, headers=HEADERS, timeout=20)
response.raise_for_status()
soup = BeautifulSoup(response.text, 'html.parser')
for tag in soup(['script', 'style', 'noscript', 'template']):
    tag.decompose()

lines = [line.strip() for line in soup.get_text('\n').splitlines() if line.strip()]
html_text = '\n'.join(dict.fromkeys(lines))
if not html_text:
    raise RuntimeError('페이지에서 공연 정보를 가져올 수 없습니다.')

IMAGE_ATTRIBUTES = ('src', 'data-src', 'data-original', 'data-lazy-src')
IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.webp', '.gif')
POSTER_PATH_PREFIXES = (
    '/play/image/large/',
    '/play/image/small/',
    '/ticketimage/notice_poster/',
)

def is_detail_image(image_url):
    parsed = urlparse(image_url)
    host = (parsed.hostname or '').lower()
    path = parsed.path.lower()

    if parsed.scheme not in {'http', 'https'}:
        return False
    if host != 'ticketimage.interpark.com':
        return False
    if path.startswith(POSTER_PATH_PREFIXES):
        return False
    return path.endswith(IMAGE_EXTENSIONS)

all_image_urls = []
detail_image_urls = []

for image in soup.find_all('img'):
    for attribute in IMAGE_ATTRIBUTES:
        candidate = image.get(attribute)
        if not candidate:
            continue

        image_url = urljoin(response.url, candidate.strip())
        if image_url not in all_image_urls:
            all_image_urls.append(image_url)
        if is_detail_image(image_url) and image_url not in detail_image_urls:
            detail_image_urls.append(image_url)

print('HTTP 상태:', response.status_code)
print('HTML 원문 길이:', len(html_text))
print('전체 이미지 URL 수:', len(all_image_urls))
for image_url in all_image_urls:
    status = 'OCR 대상' if image_url in detail_image_urls else '제외'
    print(f'- [{status}] {image_url}')
print('상세 이미지 수:', len(detail_image_urls))
if not detail_image_urls:
    print('상세 공지 이미지가 없어 HTML 정보만 사용합니다.')

HTTP 상태: 200
HTML 원문 길이: 3535
전체 이미지 URL 수: 4
- [제외] https://ticketimage.interpark.com/Play/image/large/26/26013161_p.gif
- [제외] https://static.toss.im/illusts/interpark_facepass_guideline_kr.png
- [OCR 대상] http://ticketimage.interpark.com/260131612026/09/21/068a2840.jpg
- [OCR 대상] http://ticketimage.interpark.com/260131612026/09/21/8e07d339.jpg
상세 이미지 수: 2


# 4. 상세 이미지 OCR

PaddleOCR의 한국어 PP-OCRv5 모델로 각 상세 이미지의 텍스트를 추출한다. OCR 원문은 날짜·시간·가격을 임의로 보정하지 않는다. 이미지가 없거나 일부 OCR이 실패해도 HTML 처리는 계속한다.

In [21]:
import json
from io import BytesIO

import numpy as np
from paddleocr import PaddleOCR
from PIL import Image

ocr = PaddleOCR(
    lang='korean',
    ocr_version='PP-OCRv5',
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False,
)

def collect_ocr_texts(results):
    texts = []
    for result in results:
        payload = getattr(result, 'json', result)
        payload = payload() if callable(payload) else payload
        payload = json.loads(payload) if isinstance(payload, str) else payload
        content = payload.get('res', payload) if isinstance(payload, dict) else {}
        recognized = content.get('rec_texts', []) if isinstance(content, dict) else []
        texts.extend(text for text in recognized if isinstance(text, str) and text.strip())
    return texts

def extract_image_text(image_url):
    image_response = requests.get(image_url, headers=HEADERS, timeout=30)
    image_response.raise_for_status()
    with Image.open(BytesIO(image_response.content)) as source:
        image_array = np.asarray(source.convert('RGB'))
    texts = collect_ocr_texts(ocr.predict(image_array))
    if not texts:
        raise RuntimeError('상세 이미지에서 텍스트를 찾지 못했습니다.')
    return '\n'.join(texts)

image_texts = {}
ocr_warnings = []
for image_url in detail_image_urls:
    try:
        image_texts[image_url] = extract_image_text(image_url)
    except Exception as error:
        ocr_warnings.append(f'{image_url}: {error}')

print('OCR 성공 이미지 수:', len(image_texts))
print('OCR 실패 이미지 수:', len(ocr_warnings))

Creating model: ('PP-OCRv5_server_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/content/paddlex_cache/official_models/PP-OCRv5_server_det`.
Creating model: ('korean_PP-OCRv5_mobile_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/content/paddlex_cache/official_models/korean_PP-OCRv5_mobile_rec`.
Resized image size (840x4382) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (840x26020) exceeds max_side_limit of 4000. Resizing to fit within limit.


OCR 성공 이미지 수: 2
OCR 실패 이미지 수: 0


# 5. LangChain Document

HTML과 OCR 텍스트를 각각 `Document`로 만들고 `concert_id`, `source_type`, `source_url`, `section` metadata를 보존한다.

In [22]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content=html_text,
        metadata={
            'concert_id': concert_id,
            'source_type': 'html',
            'source_url': response.url,
            'section': 'page',
        },
    )
]

for image_url in detail_image_urls:
    if image_url in image_texts:
        documents.append(Document(
            page_content=image_texts[image_url],
            metadata={
                'concert_id': concert_id,
                'source_type': 'image',
                'source_url': image_url,
                'section': 'detail_notice',
            },
        ))

print('Document 수:', len(documents))
print('source_type:', [document.metadata['source_type'] for document in documents])

Document 수: 3
source_type: ['html', 'image', 'image']


# 6. Text Splitter

`RecursiveCharacterTextSplitter`로 긴 공지를 검색 가능한 Chunk로 분할한다. 분할된 Chunk에도 원본 metadata가 유지된다.

In [23]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=['\n\n', '\n[', '\n※', '\n- ', '\n', '. ', ' ', ''],
    keep_separator='start',
)
chunks = splitter.split_documents(documents)

print(f'Document {len(documents)}개 → Chunk {len(chunks)}개')
print('metadata 보존:', all('concert_id' in chunk.metadata for chunk in chunks))

Document 3개 → Chunk 19개
metadata 보존: True


# 7. OpenAI Embedding + Chroma

`text-embedding-3-small`로 Chunk를 벡터화하고 Chroma에 저장한다. 모든 검색에는 현재 URL에서 추출한 `concert_id` filter를 적용하여 다른 공연의 정보가 섞이지 않게 한다.

In [24]:
import chromadb
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
vector_store = Chroma(
    client=chromadb.EphemeralClient(),
    collection_name='concert_ticket_guide_notebook',
    embedding_function=embeddings,
)
vector_store.add_documents(chunks)

print('저장된 Chunk 수:', vector_store._collection.count())

저장된 Chunk 수: 38


# 8. Query Understanding + Structured Output

대표 질문을 Chat Model의 Structured Output으로 예매 유형·수령 방식·필요 주제로 분석한다. 질문에 명시되지 않은 조건은 만들지 않는다.


In [26]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate

class QueryAnalysis(BaseModel):
    booking_type: Literal['fanclub_presale', 'general_sale', 'unknown'] = 'unknown'
    ticket_delivery: Literal['delivery', 'onsite', 'unknown'] = 'unknown'
    needed_topics: list[str] = Field(default_factory=list)
    search_query: str

query_prompt = ChatPromptTemplate.from_messages([
    ('system', '콘서트 예매 질문을 검색 조건으로 구조화하세요. 질문에 명시된 조건만 사용하고 날짜, 시간, 가격을 만들지 마세요. search_query에는 검색에 필요한 핵심 한국어 키워드를 작성하세요.'),
    ('human', '{question}'),
])
query_chain = query_prompt | model.with_structured_output(QueryAnalysis)

# 실제 서비스에서는 사용자 질문을 입력받지만,
# 이 노트북은 보고서 재현성을 위해 대표 질문을 고정한다.
question = '본인확인이 가능한 신분증이 뭔가요'

print('대표 질문:', question)
query_analysis = query_chain.invoke({'question': question})
print(query_analysis.model_dump_json(indent=2))


대표 질문: 본인확인이 가능한 신분증이 뭔가요
{
  "booking_type": "unknown",
  "ticket_delivery": "unknown",
  "needed_topics": [
    "신분증 종류",
    "본인확인 방법"
  ],
  "search_query": "신분증 본인확인"
}


# 9. Candidate Retrieval / Reranking

Chroma의 의미 기반 검색 결과와 핵심어 점수를 결합해 **후보 Context**를 찾는다.  
이 단계의 결과는 아직 최종 근거가 아니다. 검색어가 겹치거나 의미가 비슷하다는 이유만으로 질문에 직접 답하지 않는 Chunk가 포함될 수 있으므로, 다음 단계에서 Evidence Filter로 한 번 더 검증한다.


In [27]:
import math
import re

VECTOR_CANDIDATE_K = min(10, len(chunks))
FINAL_RESULT_K = 5
MAX_IMAGE_RESULTS = 2
SEARCH_STOPWORDS = {
    '공연', '티켓', '안내', '정보', '관련', '방법', '일정',
    '언제', '어떻게', '알려줘', '해줘', '하는', '되어', '있어',
}
KOREAN_SUFFIXES = (
    '으로부터', '에서부터', '까지는', '부터는', '에서는',
    '으로', '에서', '부터', '까지', '하고',
    '은', '는', '이', '가', '을', '를', '의', '에', '로',
)

def document_key(document):
    return (
        document.metadata.get('source_url', ''),
        document.page_content,
    )

def extract_search_terms(*texts):
    terms = set()
    for text in texts:
        for token in re.findall(r'[가-힣A-Za-z0-9]+', text.lower()):
            candidates = {token}
            for suffix in KOREAN_SUFFIXES:
                if token.endswith(suffix) and len(token) > len(suffix) + 1:
                    candidates.add(token[:-len(suffix)])
            terms.update(
                candidate for candidate in candidates
                if len(candidate) >= 2 and candidate not in SEARCH_STOPWORDS
            )
    return sorted(terms)

def retrieve_documents(question_text, analysis, show_debug=True):
    search_query = analysis.search_query.strip() or question_text
    topic_text = ' '.join(analysis.needed_topics)
    search_terms = extract_search_terms(question_text, search_query, topic_text)

    semantic_results = vector_store.similarity_search_with_score(
        search_query,
        k=VECTOR_CANDIDATE_K,
        filter={'concert_id': concert_id},
    )
    semantic_ranks = {
        document_key(document): rank
        for rank, (document, _) in enumerate(semantic_results, start=1)
    }

    document_frequency = {
        term: sum(term in chunk.page_content.lower() for chunk in chunks)
        for term in search_terms
    }
    lexical_scores = {}
    matched_terms = {}
    for chunk in chunks:
        key = document_key(chunk)
        content = chunk.page_content.lower()
        matches = [term for term in search_terms if term in content]
        matched_terms[key] = matches
        lexical_scores[key] = sum(
            math.log((len(chunks) + 1) / (document_frequency[term] + 1)) + 1
            for term in matches
        )

    max_lexical_score = max(lexical_scores.values(), default=0)
    candidate_documents = {
        document_key(document): document
        for document, _ in semantic_results
    }
    candidate_documents.update({
        document_key(chunk): chunk
        for chunk in chunks
        if lexical_scores[document_key(chunk)] > 0
    })

    ranked_candidates = []
    for key, document in candidate_documents.items():
        semantic_rank = semantic_ranks.get(key)
        semantic_score = (
            (VECTOR_CANDIDATE_K - semantic_rank + 1) / VECTOR_CANDIDATE_K
            if semantic_rank is not None else 0
        )
        lexical_score = (
            lexical_scores.get(key, 0) / max_lexical_score
            if max_lexical_score else 0
        )
        html_bonus = (
            0.10
            if document.metadata.get('source_type') == 'html'
            and lexical_score > 0
            else 0
        )
        combined_score = (
            0.35 * semantic_score
            + 0.55 * lexical_score
            + html_bonus
        )
        ranked_candidates.append(
            (combined_score, semantic_rank, document)
        )

    ranked_candidates.sort(
        key=lambda item: (
            -item[0],
            item[2].metadata.get('source_type') != 'html',
            item[1] if item[1] is not None else VECTOR_CANDIDATE_K + 1,
        )
    )

    found_documents = []
    retrieval_debug = {}
    image_result_count = 0

    for combined_score, semantic_rank, document in ranked_candidates:
        if document.metadata.get('source_type') == 'image':
            if image_result_count >= MAX_IMAGE_RESULTS:
                continue
            image_result_count += 1

        found_documents.append(document)
        retrieval_debug[document_key(document)] = {
            'score': combined_score,
            'vector_rank': semantic_rank,
            'matched_terms': matched_terms.get(document_key(document), []),
        }

        if len(found_documents) == FINAL_RESULT_K:
            break

    if show_debug:
        print('벡터 검색어:', search_query)
        print('핵심어:', search_terms)
        for index, document in enumerate(found_documents, start=1):
            debug = retrieval_debug[document_key(document)]
            print(
                f"[{index}] score={debug['score']:.3f} | "
                f"vector_rank={debug['vector_rank']} | "
                f"keywords={debug['matched_terms']} | "
                f"{document.metadata['source_type']} | "
                f"{document.metadata['source_url']}"
            )
            print(
                document.page_content[:300].replace('\n', ' '),
                '\n'
            )

    return found_documents, retrieval_debug, search_query, search_terms

retrieved_documents, retrieval_debug, search_query, search_terms = retrieve_documents(
    question,
    query_analysis,
    show_debug=True,
)


벡터 검색어: 신분증 본인확인
핵심어: ['가능한', '뭔가요', '본인확인', '본인확인이', '신분증', '신분증이', '종류']
[1] score=0.865 | vector_rank=2 | keywords=['가능한', '본인확인', '본인확인이', '신분증'] | image | http://ticketimage.interpark.com/260131612026/09/21/8e07d339.jpg
-지참하신 물품은 입장시작 시간 전 외부에 마련된 물품보관소(유료)에 보관 가능하오나,정해진 공간 내 운영되므로 -얼굴패스를 사전 등록하신경우,별도의 티켓 검표 및 본인 확인 배부됩니다 -얼굴패스를사전등록하였으나인식오류등현장에서 본인 확인이불가한경우지정유효신분증과모바일 티켓을 CS부스는 공연시작7시간 전부터 운영됩니다. -얼굴패스 등록을 원하지 않을 경우 별도로 마련된 IN-PASS부스를 방문하시어 본인 확인 및 티켓 검표 부탁드립니다. IN-PASS부스는 공연 시작 7시간 전부터 운영되며,본인 확인을 위해지정유효신분증과 모바일 티 

[2] score=0.860 | vector_rank=5 | keywords=['가능한', '본인확인', '본인확인이', '신분증'] | html | https://nol.yanolja.com/ticket/products/26013161
※ 티켓 예매 시 공연 안내사항에 동의한 것으로 간주하며, 본 내용은 공연 상황에 따라 추가/변경될 수 있습니다. 공연 관람에 지장이나 불이익을 받지 않도록 관람 전 반드시 공연 안내사항을 재확인 바랍니다. ※ 공연장에서 모바일 티켓 열람 시, 데이터 트래픽에 의해 원활하지 않을 수 있으니 공연장 출발 전 모바일 티켓 사전 열람 및 저장하신 후 공연장에 도착 부탁드립니다. ※ 원활한 예매를 위해 예매 전 본인인증을 완료해 주시기 바랍니다. - ‘마이>NOL 계정관리>본인 인증’를 통해 언제든지 본인인증을 할 수 있으며, 최종 본인인 

[3] score=0.690 | vector_rank=7 | ke

# 10. Evidence Filter — 검색 결과가 진짜 질문의 근거인가?

Retriever의 상위 결과는 **관련 후보**일 뿐, 모두 정답 근거라고 볼 수 없다.
예를 들어 `선예매`라는 단어가 들어 있어도 단순히 일반예매 제한을 설명하는 문장은
`선예매 자격 조건`의 직접 근거가 아닐 수 있다.

따라서 후보 Chunk를 최종 LLM에 그대로 넣지 않고,
질문에 **직접 답할 수 있는 사실을 포함하는지** 한 번 더 판별한다.

판별 원칙:

- 같은 단어가 등장한다는 이유만으로 통과시키지 않는다.
- 질문의 핵심 사실 또는 조건을 직접 설명해야 한다.
- 여러 조건이 있는 질문이라면 그중 하나에 직접 답하는 Chunk는 통과할 수 있다.
- 판단이 애매하거나 간접적인 내용뿐이면 제외한다.
- 최종 Answer Chain에는 통과한 Context만 전달한다.

이 단계 역시 LLM을 사용하지만, 자유 답변이 아니라 `supported: bool` 형태의
Structured Output으로 역할을 **근거 판별**에만 제한한다.


In [28]:
class EvidenceDecision(BaseModel):
    document_id: int
    supported: bool
    reason: str

class EvidenceFilterResponse(BaseModel):
    decisions: list[EvidenceDecision] = Field(default_factory=list)

evidence_prompt = ChatPromptTemplate.from_messages([
    ('system', '''당신은 RAG의 Evidence Filter입니다.
사용자 질문에 대해 각 DOCUMENT가 직접적인 답변 근거인지 판별하세요.

판단 기준:
1. 질문과 비슷한 단어나 주제가 등장한다는 사실만으로 supported=true로 판단하지 마세요.
2. DOCUMENT가 질문의 핵심 사실, 조건, 일정, 준비물 중 하나를 직접 설명해야 합니다.
3. 여러 조건이 포함된 질문에서는 그중 일부에 직접 답하는 DOCUMENT도 supported=true가 될 수 있습니다.
4. 많은 추론이 필요하거나 주변 정보만 제공하면 supported=false입니다.
5. DOCUMENT에 없는 사실을 만들어 판단하지 마세요.
6. 모든 DOCUMENT에 대해 정확히 한 개의 decision을 반환하세요.'''),
    ('human', '''[QUESTION]
{question}

[CANDIDATE DOCUMENTS]
{documents}'''),
])

evidence_chain = (
    evidence_prompt
    | model.with_structured_output(EvidenceFilterResponse)
)

def format_candidate_documents(documents):
    return '\n\n'.join(
        f"[DOCUMENT {index}]\n"
        f"source_type: {document.metadata.get('source_type')}\n"
        f"source_url: {document.metadata.get('source_url')}\n"
        f"section: {document.metadata.get('section')}\n"
        f"content: {document.page_content}"
        for index, document in enumerate(documents, start=1)
    )

def filter_evidence(question_text, candidate_documents, show_debug=True):
    if not candidate_documents:
        return [], []

    judgement = evidence_chain.invoke({
        'question': question_text,
        'documents': format_candidate_documents(candidate_documents),
    })

    decision_map = {
        decision.document_id: decision
        for decision in judgement.decisions
        if 1 <= decision.document_id <= len(candidate_documents)
    }

    supported_documents = []
    decisions = []

    # 판정이 누락된 문서는 보수적으로 unsupported 처리
    for document_id, document in enumerate(candidate_documents, start=1):
        decision = decision_map.get(
            document_id,
            EvidenceDecision(
                document_id=document_id,
                supported=False,
                reason='Evidence Filter가 이 문서에 대한 판정을 반환하지 않아 제외',
            ),
        )
        decisions.append(decision)

        if decision.supported:
            supported_documents.append(document)

        if show_debug:
            status = 'PASS' if decision.supported else 'REJECT'
            print(
                f"[{status}] candidate #{document_id} | "
                f"{document.metadata.get('source_type')} | "
                f"{decision.reason}"
            )
            print(document.page_content[:220].replace('\n', ' '), '\n')

    print(
        f"Evidence Filter: 후보 {len(candidate_documents)}개 → "
        f"최종 근거 {len(supported_documents)}개"
    )
    return supported_documents, decisions

evidence_documents, evidence_decisions = filter_evidence(
    question,
    retrieved_documents,
    show_debug=True,
)


[PASS] candidate #1 | image | DOCUMENT 1 directly lists the valid forms of identification for personal verification, including 주민등록증, 운전면허증, 여권, and 청소년증, which answers the question about what constitutes a valid ID for personal verification.
-지참하신 물품은 입장시작 시간 전 외부에 마련된 물품보관소(유료)에 보관 가능하오나,정해진 공간 내 운영되므로 -얼굴패스를 사전 등록하신경우,별도의 티켓 검표 및 본인 확인 배부됩니다 -얼굴패스를사전등록하였으나인식오류등현장에서 본인 확인이불가한경우지정유효신분증과모바일 티켓을 CS부스는 공연시작7시간 전부터 운영됩니다. -얼굴패스 등록을 원하지 않을 경우 별도로 마련된 IN-PASS부스를 방 

[REJECT] candidate #2 | html | DOCUMENT 2 discusses the importance of personal identification but does not specify what types of identification are considered valid.
※ 티켓 예매 시 공연 안내사항에 동의한 것으로 간주하며, 본 내용은 공연 상황에 따라 추가/변경될 수 있습니다. 공연 관람에 지장이나 불이익을 받지 않도록 관람 전 반드시 공연 안내사항을 재확인 바랍니다. ※ 공연장에서 모바일 티켓 열람 시, 데이터 트래픽에 의해 원활하지 않을 수 있으니 공연장 출발 전 모바일 티켓 사전 열람 및 저장하신 후 공연장에 도착 부탁드립니다. ※ 원활한 예매를  

[REJECT] candidate #3 | image | DOCUMENT 3 mentions the need for identification but does not provide specific examples of valid IDs.
[SOUND CHECK

# 11. Prompt Template + RAG Answer Chain

Evidence Filter를 통과한 Context만 최종 답변 모델에 전달한다.  
근거가 0개라면 LLM에게 억지로 답을 만들게 하지 않고 코드에서 즉시 `예매 페이지에서 확인할 수 없습니다.`를 반환한다.


In [30]:
NO_EVIDENCE_MESSAGE = '예매 페이지에서 확인할 수 없습니다.'

class TicketGuideResponse(BaseModel):
    summary: str
    schedule: list[str] = Field(default_factory=list)
    requirements: list[str] = Field(default_factory=list)
    ticket_info: list[str] = Field(default_factory=list)
    warnings: list[str] = Field(default_factory=list)
    sources: list[str] = Field(default_factory=list)

answer_prompt = ChatPromptTemplate.from_messages([
    ('system', '''당신은 콘서트 예매 안내 도우미입니다.
REFERENCE DOCUMENTS는 Retriever 후보 중 Evidence Filter를 통과한 문서입니다.
그래도 각 문장의 사실은 REFERENCE DOCUMENTS 원문으로 확인하며 답하세요.

규칙:
1. REFERENCE DOCUMENTS에 있는 원문만 공연 정보의 근거로 사용하세요.
2. HTML과 OCR에 동일한 정보가 있으면 source_type이 html인 원문을 우선하세요.
3. Context에 없는 날짜, 시간, 가격, 정책은 추측하거나 수정하지 마세요.
4. 질문과 직접 관련된 근거가 있으면 summary에서 먼저 질문에 직접 답하세요.
5. 질문의 일부만 확인 가능하면 확인 가능한 부분만 답하고, 확인할 수 없는 부분을 구분해서 말하세요.
6. requirements, schedule, ticket_info, warnings에 내용을 작성했다면
   summary 전체를 "예매 페이지에서 확인할 수 없습니다."라고 쓰지 마세요.
7. 질문과 직접 관련 없는 정보는 출력하지 마세요.
8. sources에는 실제 답변 근거로 사용한 source_url만 넣으세요.'''),
    ('human', '''[USER QUESTION]
{question}

[USER CONTEXT]
{user_context}

[REFERENCE DOCUMENTS]
{context}'''),
])

answer_chain = (
    answer_prompt
    | model.with_structured_output(TicketGuideResponse)
)

def format_context(found_documents):
    # 동일 정보가 충돌하면 HTML을 우선하도록 HTML 문서를 앞에 둔다.
    ordered = sorted(
        found_documents,
        key=lambda doc: doc.metadata['source_type'] != 'html'
    )
    return '\n\n'.join(
        f"source_type: {doc.metadata['source_type']}\n"
        f"source_url: {doc.metadata['source_url']}\n"
        f"section: {doc.metadata['section']}\n"
        f"content: {doc.page_content}"
        for doc in ordered
    )

def generate_answer(question_text, analysis, evidence_docs):
    # 근거가 하나도 없으면 LLM 생성 단계 자체를 건너뛴다.
    if not evidence_docs:
        return TicketGuideResponse(summary=NO_EVIDENCE_MESSAGE)

    context = format_context(evidence_docs)
    result = answer_chain.invoke({
        'question': question_text,
        'user_context': analysis.model_dump_json(),
        'context': context,
    })

    # Structured Output은 타입은 보장하지만 필드 간 의미 일관성까지 보장하지 않는다.
    # 근거가 있는데 세부 필드까지 생성하면서 summary만 '근거 없음'으로 나오는 모순을 감지한다.
    has_detail = any([
        result.schedule,
        result.requirements,
        result.ticket_info,
        result.warnings,
    ])
    if has_detail and result.summary.strip() == NO_EVIDENCE_MESSAGE:
        raise ValueError(
            '응답 의미 불일치: 근거 기반 세부 항목이 존재하지만 '
            'summary가 근거 없음으로 생성되었습니다.'
        )

    return result

result = generate_answer(
    question,
    query_analysis,
    evidence_documents,
)


### Prompt / Context 설계 의도

| 설계 요소 | 적용 내용 | 이유 |
|---|---|---|
| Candidate Retrieval | 의미 검색 + 핵심어 + HTML bonus로 후보를 넓게 수집 | 정답 근거를 너무 일찍 놓치는 것을 방지 |
| Evidence Filter | 후보가 질문에 **직접 답하는 근거인지** 별도 판별 | 관련 단어만 있는 엉뚱한 Chunk가 최종 Context에 들어가는 것을 방지 |
| Role | `콘서트 예매 안내 도우미` | 일반 지식보다 예매 공지 해석에 역할 제한 |
| Grounding | Evidence Filter를 통과한 문서만 `REFERENCE DOCUMENTS`로 전달 | 검색 후보 전체를 근거로 오인하는 위험 감소 |
| HTML 우선 | HTML과 OCR이 충돌하면 HTML 우선 | OCR 문자·숫자 오인식 가능성 완화 |
| No-evidence 처리 | 최종 근거가 0개면 Answer LLM을 호출하지 않고 고정 메시지 반환 | 근거 없는 생성 자체를 차단 |
| Structured Output | `summary`, `schedule`, `requirements`, `ticket_info`, `warnings`, `sources` | 결과 비교·후처리가 쉬운 일정한 형태 보장 |
| 의미 일관성 검사 | 근거 기반 세부 항목이 있는데 summary만 `확인 불가`이면 오류 처리 | 타입 검증만으로 잡히지 않는 필드 간 모순 탐지 |
| Few-shot 미사용 | 공연별 날짜·가격 예시를 Prompt에 고정하지 않음 | 예시의 사실값이 다른 공연 답변에 섞일 위험 감소 |


# 12. 대표 질문 최종 결과

아래 결과는 **Candidate Retrieval → Evidence Filter**를 통과한 Context만 사용해 생성된다.


In [31]:
print(json.dumps(result.model_dump(), ensure_ascii=False, indent=2))

if ocr_warnings:
    print('\n주의: 상세 공지 이미지 분석에 실패해 일부 안내가 누락되었을 수 있습니다.')

{
  "summary": "본인확인이 가능한 신분증은 다음과 같습니다: 대한민국 국적자는 주민등록증, 운전면허증, 여권, 청소년증(만 9세 이상부터 주민센터 발급), 모바일 신분증(정부24, 행정안전부, 경찰청, PASS 발급)입니다. 대한민국 국적 외에는 여권과 외국인등록증이 유효한 신분증으로 인정됩니다.",
  "schedule": [],
  "requirements": [],
  "ticket_info": [],
  "warnings": [],
  "sources": [
    "http://ticketimage.interpark.com/260131612026/09/21/8e07d339.jpg"
  ]
}


# 13. 입력을 바꾼 3회 실행 비교

같은 공연 URL과 같은 Vector Store를 유지한 채 **질문만 변경**한다.
각 질문마다 다음 세 단계를 비교한다.

`Candidate Retrieval → Evidence Filter → Answer`

즉, 단순히 상위 검색 결과가 달라지는지만 보는 것이 아니라
**검색 후보 중 실제 질문의 근거로 인정된 Context가 무엇인지**까지 확인한다.

비교 입력:

1. `선예매 하려면 조건이 있나요?`
2. `휠체어석은 어떻게 예매하고 현장에서 뭘 준비해야 해?`
3. `일반예매 후 현장에서 티켓을 받을 건데 준비물이 뭐야?`


In [32]:
import pandas as pd
from IPython.display import display

test_questions = [
    '선예매 하려면 조건이 있나요?',
    '휠체어석은 어떻게 예매하고 현장에서 뭘 준비해야 해?',
    '일반예매 후 현장에서 티켓을 받을 건데 준비물이 뭐야?',
]

comparison_rows = []

for test_question in test_questions:
    if test_question == question:
        analysis = query_analysis
        candidate_docs = retrieved_documents
        passed_docs = evidence_documents
        answer = result
    else:
        analysis = query_chain.invoke({'question': test_question})
        candidate_docs, _, _, _ = retrieve_documents(
            test_question,
            analysis,
            show_debug=False,
        )
        passed_docs, _ = filter_evidence(
            test_question,
            candidate_docs,
            show_debug=False,
        )
        answer = generate_answer(
            test_question,
            analysis,
            passed_docs,
        )

    candidate_preview = ' | '.join(
        f"{doc.metadata['source_type']}: "
        f"{doc.page_content[:90].replace(chr(10), ' ')}"
        for doc in candidate_docs[:2]
    )

    evidence_preview = ' | '.join(
        f"{doc.metadata['source_type']}: "
        f"{doc.page_content[:110].replace(chr(10), ' ')}"
        for doc in passed_docs[:2]
    ) or '(직접 근거 없음)'

    key_output = ' / '.join(
        [answer.summary]
        + answer.schedule[:2]
        + answer.requirements[:2]
        + answer.ticket_info[:2]
    )

    comparison_rows.append({
        '입력': test_question,
        'Query Understanding': (
            f"booking={analysis.booking_type}, "
            f"delivery={analysis.ticket_delivery}, "
            f"topics={analysis.needed_topics}"
        ),
        '후보 수': len(candidate_docs),
        '검증 근거 수': len(passed_docs),
        'Candidate Context 상위 2개': candidate_preview,
        'Evidence Context 상위 2개': evidence_preview,
        '핵심 출력': key_output,
    })

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)


Evidence Filter: 후보 5개 → 최종 근거 1개
Evidence Filter: 후보 5개 → 최종 근거 2개
Evidence Filter: 후보 5개 → 최종 근거 1개


,입력,Query Understanding,후보 수,검증 근거 수,Candidate Context 상위 2개,Evidence Context 상위 2개,핵심 출력
0,선예매 하려면 조건이 있나요?,"booking=fanclub_presale, delivery=unknown, top...",5,1,"image: 공연장 입장이 가능하며,IN-PASS부스 방문시 반드시 본인 확인이 가...",html: [예매 안내] ※ 본 공연의 예매는 아래 절차로 이루어집니다. 국내 페이...,"선예매를 하려면 MOA Membership에 가입해야 하며, 예매 전 NOL 회원정..."
1,휠체어석은 어떻게 예매하고 현장에서 뭘 준비해야 해?,"booking=unknown, delivery=onsite, topics=['휠체어...",5,2,html: [휠체어석 예매 안내] - 휠체어석 구매는 2026년 10월 7일(수) ...,html: [휠체어석 예매 안내] - 휠체어석 구매는 2026년 10월 7일(수) ...,휠체어석은 2026년 10월 7일(수) 오전 9시(KST)부터 NOL 고객센터(15...
2,일반예매 후 현장에서 티켓을 받을 건데 준비물이 뭐야?,"booking=general_sale, delivery=onsite, topics=...",5,1,"image: ※멤버십 선예매에서1회차공연을 예매한경우,1회차 공연 일반예매참여불가 ...",html: [휠체어석 예매 안내] - 휠체어석 구매는 2026년 10월 7일(수) ...,현장에서 티켓을 수령하기 위해서는 본인 신분증(실물)을 준비해야 합니다.


### 비교에서 확인할 점

- 같은 공연이어도 질문이 바뀌면 Query Understanding 결과가 달라진다.
- Retriever는 정답을 확정하지 않고 **후보 Context**를 넓게 찾는다.
- Evidence Filter는 그중 질문에 직접 답하는 Chunk만 남긴다.
- 최종 Answer Chain에는 통과한 Context만 들어간다.
- 통과한 근거가 하나도 없으면 LLM에게 추측을 시키지 않고 `예매 페이지에서 확인할 수 없습니다.`를 반환한다.

따라서 이 실험의 핵심은 단순히 답변 문장이 달라지는 것이 아니라,
**질문별로 최종 모델에 투입되는 Context가 선택·검증되는 과정 자체가 달라진다는 점**이다.


# 14. LangChain 컴포넌트 활용 — 왜 이 자리에 사용했는가

| 컴포넌트 | 사용 이유 | 없었다면 무엇이 어려운가 |
|---|---|---|
| `Document` | HTML과 OCR을 같은 문서 단위로 만들고 `source_type`, `source_url`, `concert_id` metadata를 보존 | 출처가 HTML인지 이미지인지 구분하거나 답변 근거를 추적하기 어려움 |
| `RecursiveCharacterTextSplitter` | 긴 공지를 검색 가능한 Chunk로 분할하면서 metadata 유지 | 전체 공지를 매번 LLM에 넣어 관련 없는 정보까지 Context에 포함됨 |
| `OpenAIEmbeddings` | 사용자의 자연어 질문과 의미적으로 가까운 Chunk 탐색 | 질문과 공지의 표현이 다르면 단순 키워드만으로 찾기 어려움 |
| `Chroma` | Embedding된 Chunk를 저장하고 현재 `concert_id` 범위에서 유사도 검색 | 질문마다 전체 Chunk를 직접 비교해야 함 |
| Hybrid Retrieval/Reranking | Chroma 벡터 순위 + 핵심어 점수 + HTML bonus를 조합 | 벡터 검색만으로 날짜·가격·고유명사처럼 정확한 단어가 중요한 Chunk가 밀릴 수 있음 |
| **Evidence Filter Chain** | 검색 후보가 질문의 **직접 근거인지** Structured Output으로 재판별 | 관련 단어만 있는 Chunk를 정답 근거로 오인해 최종 답변이 왜곡될 수 있음 |
| `ChatPromptTemplate` | 역할, 사용자 질문, 사용자 Context, 검증된 검색 문서를 일정한 형식으로 전달 | 호출마다 Prompt 구조와 제약이 달라져 일관성이 낮아짐 |
| LCEL `prompt | model` | Query Understanding, Evidence Filter, Answer 단계를 명시적으로 연결 | Prompt 생성과 모델 호출을 매번 수동으로 관리해야 함 |
| `with_structured_output` | 질문 분석·근거 판별·최종 답변을 Pydantic 구조로 고정 | 자유 텍스트라 다음 단계에서 안정적으로 사용하거나 비교하기 어려움 |

> 이 노트북은 `vector_store.as_retriever()`를 그대로 사용하는 대신
> `similarity_search_with_score()` 기반 후보 검색 뒤에 사용자 정의 Reranking과
> **LLM Evidence Filter**를 적용한다.


# 15. 한계와 개선 방향

### 1. 긴 상세 이미지 OCR

실제 실행에서 상세 이미지의 세로 길이가 `4000px`을 넘어
PaddleOCR이 자동 축소하는 로그가 발생했다.
현재 테스트에서는 **2개 이미지 모두 OCR에 성공**했지만,
작은 글씨의 인식 정확도까지 정량적으로 검증한 것은 아니다.

**개선:** 긴 이미지를 구간별로 분할한 OCR 결과와 현재 방식의 정확도를 비교한다.

### 2. OCR 오인식 가능성

이미지 OCR은 영문, 숫자, 띄어쓰기 등을 잘못 읽을 수 있다.
특히 날짜·시간·가격을 임의로 자동 보정하면
오히려 사실과 다른 값을 만들 위험이 있다.

**현재 대응:** 동일 정보가 HTML과 OCR에 있으면 HTML을 우선하고,
OCR의 숫자·날짜를 코드에서 추측해 수정하지 않는다.

### 3. Retriever의 후보가 곧 근거는 아님 — 테스트에서 발견한 문제

`선예매 조건`처럼 짧은 질문에서는 `선예매`라는 단어가 포함됐다는 이유로
일반예매 제한, M&G 안내 등 **주제는 비슷하지만 질문에 직접 답하지 않는 Chunk**도
상위 후보에 포함될 수 있었다.

검색 후보를 그대로 Answer LLM에 전달하면 잘못된 Context를 근거로 답할 위험이 있어
이번 버전에서는 **Candidate Retrieval과 Answer 사이에 Evidence Filter를 추가**했다.

### 4. Evidence Filter도 완벽한 판별기는 아님

Evidence Filter 역시 LLM 기반이므로 직접 근거를 잘못 통과시키는 false positive,
실제 근거를 제외하는 false negative가 발생할 수 있다.

**개선:** 질문별 정답 근거 Chunk를 표시한 Golden Set을 만들고
Evidence Filter 정확도, Chunk 크기, Top-K, 가중치 및 별도 Reranker를 함께 비교한다.

### 5. Retrieval/Reranking 가중치가 경험적으로 설정됨

현재 점수는 `semantic 0.35 + lexical 0.55 + HTML bonus 0.10`으로 구성했다.
이 값은 소규모 테스트를 통해 정한 휴리스틱이며,
충분한 평가 데이터로 최적화한 값은 아니다.

**개선:** Golden Set에서 Recall@K와 최종 근거 선택 정확도를 측정해 가중치를 조정한다.

### 6. NOL Ticket 페이지 구조 의존

현재 Loader와 상세 이미지 탐지 규칙은 NOL Ticket 상품 페이지 구조를 기준으로 한다.
사이트의 HTML 또는 이미지 경로 정책이 바뀌면 수집 규칙도 수정해야 한다.

**개선:** Loader 실패를 감지하는 검증 로직과 페이지 구조별 fallback을 추가한다.
